In [ ]:
import json
import os

pasta_dados = "/content/atelie-generativo-pequi-do-cerrado/dados"
txt_path = f"{pasta_dados}/legendas.txt"
jsonl_path = f"{pasta_dados}/metadata.jsonl"

with open(txt_path, 'r', encoding='utf-8') as f_in, open(jsonl_path, 'w', encoding='utf-8') as f_out:
    for linha in f_in:
        if ":" in linha:
            arquivo, legenda = linha.strip().split(":", 1)
            # Cria a estrutura que a IA entende
            linha_json = json.dumps({"file_name": arquivo.strip(), "text": legenda.strip()})
            f_out.write(linha_json + "\n")

In [ ]:
!pip -q install diffusers transformers accelerate peft datasets
!git clone https://github.com/huggingface/diffusers

from huggingface_hub import login
login()

In [ ]:


%cd /content/diffusers
!pip install -e .
!pip install -q "torchao>=0.16.0"

# Entrando na pasta do script
%cd /content/diffusers/examples/text_to_image

# Iniciando o treinamento primeiro treinamento 
!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/content/atelie-generativo-pequi-do-cerrado/dados" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1500 \
  --learning_rate=1e-4 \
  --lr_scheduler="cosine" \
  --rank=8 \
  --mixed_precision="fp16" \
  --checkpointing_steps=500 \
  --validation_prompt="estilo_rupestre, uma pintura de um animal com chifres em uma parede de rocha" \
  --output_dir="/content/lora_saida" \
  --push_to_hub \
  --hub_model_id="LuisCurado/lora-estilo-rupestre"

# Iniciando o SEGUNDO treinamento (Versão 2)
!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/content/atelie-generativo-pequi-do-cerrado/dados" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1000 \
  --learning_rate=5e-5 \
  --lr_scheduler="cosine" \
  --rank=4 \
  --mixed_precision="fp16" \
  --checkpointing_steps=500 \
  --validation_prompt="estilo_rupestre, uma pintura de um animal com chifres em uma parede de rocha" \
  --output_dir="/content/lora_saida_v2" \
  --push_to_hub \
  --hub_model_id="LuisCurado/lora-estilo-rupestre-v2"
